# SGLang 异步并发入门：`async_generate(...)` 最小实践

**定位**：这是一篇入门向 SGLang Notebook，聚焦 `engine.async_generate(...)` 的异步并发调用方式，帮助你理解“单次请求 API”到“并发请求 gather”的最小迁移路径。

**选题来源（UT -> 教程化）**：本主题参考 `sglang/test/registered/models/test_generation_models.py` 的生成模型验证思路，并对齐 `sglang/examples/runtime/engine/offline_batch_inference_async.py` 的异步 API 用法。

**开源说明**：本文为原创教学示例，可用于学习与社区分享（如 Gitee）；请同时遵守所使用模型与第三方依赖的许可证。

---

## 版本说明（请先确认）


建议先确认本机版本：

```bash
python -c "import sglang as sgl; print(getattr(sgl, '__version__', 'unknown'))"
```

官方文档：<https://docs.sglang.io/>

---

## 适用显卡与环境建议

| 项目 | 建议 |
|---|---|
| 显卡类型 | NVIDIA GPU（单卡即可） |
| 建议显存 | 8GB 及以上（推荐 12GB+） |
| 驱动 / CUDA | 与本机 `torch` 匹配 |
| Python | 3.8+ |
| 系统 | Linux 优先（Windows 建议 WSL2） |
| 默认模型 | `Qwen/Qwen2.5-1.5B-Instruct` |

> 说明：异步并发不是无限吞吐，仍需结合显存与 token 池设置做保守调参。

---

## 完整依赖安装命令（终端执行）

```bash
# 0) 创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate

# 1) 升级 pip
python -m pip install --upgrade pip

# 2) 按你的 CUDA 版本安装 torch（下面仅示例）
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 3) 安装 SGLang + OpenAI SDK
pip install -U sglang openai

# 4) 运行本 Notebook 需要 Jupyter（任选其一）
pip install jupyter
```

---

## 简单使用说明

1. 先在终端执行上面的安装命令，并激活虚拟环境。
2. 在本jupyter notebook所在路径下，启动 Jupyter：jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
3. 按顺序运行单元格（环境自检 -> 异步并发 demo -> 资源释放）。
4. 首次运行会联网拉模型。

---

In [1]:
# 环境自检：Python / torch / CUDA / sglang 版本

import sys  # 读取 Python 版本

print("Python:", sys.version)

try:
    import torch  # 检查 torch 是否可导入
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch 检查失败:", repr(e))

try:
    import sglang as sgl  # 检查 sglang 是否可导入
    print("sglang:", getattr(sgl, "__version__", "unknown"))
except Exception as e:
    print("sglang 检查失败:", repr(e))

Python: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


torch: 2.9.1+cu129
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


sglang: 0.5.9


## 概念速读：为什么要用 `async_generate`？

- 同步 `generate` 更直观，但在“很多短请求”场景下会更容易串行等待。
- `async_generate` 可以把多个请求交给事件循环并发调度。
- 本例通过 `asyncio.gather(...)` 一次提交多条请求，观察端到端耗时。
- 这只是入门演示，不等同于严格性能 benchmark。

---

In [2]:
# Demo 主体：async_generate 并发请求最小示例

import asyncio  # 异步任务调度
import time  # 统计耗时

import sglang as sgl  # 导入 SGLang

MODEL_PATH = "Qwen/Qwen2.5-1.5B-Instruct"  # 入门小模型

# 初始化 Engine（保持参数保守，优先可跑）
engine = sgl.Engine(
    model_path=MODEL_PATH,
    tp_size=1,
    mem_fraction_static=0.70,
    max_running_requests=8,
    max_total_tokens=8192,
)

# 准备一组短请求
prompts = [
    "用一句话解释什么是并发。",
    "用一句话解释什么是异步。",
    "给出一个 async/await 的直觉比喻。",
    "一句话说明为什么大模型服务常见请求排队。",
]

sampling_params = {
    "temperature": 0.2,
    "top_p": 0.9,
    "max_new_tokens": 64,
}


async def one_request(prompt):
    """单请求异步生成。"""
    return await engine.async_generate(prompt=prompt, sampling_params=sampling_params)


async def run_async_batch():
    """并发提交请求并汇总结果。"""
    t0 = time.time()
    tasks = [asyncio.create_task(one_request(p)) for p in prompts]
    results = await asyncio.gather(*tasks)
    elapsed = time.time() - t0
    return results, elapsed


# 在 Jupyter 中可直接使用 await
results, elapsed = await run_async_batch()
print(f"并发请求完成，总耗时: {elapsed:.2f}s")

for i, item in enumerate(results):
    text = item.get("text", str(item)) if isinstance(item, dict) else str(item)
    print(f"\n===== 样本 {i} =====")
    print(text[:600])

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[2026-04-06 08:50:24] INFO server_args.py:1835: Attention backend not specified. Use fa3 backend by default.


[2026-04-06 08:50:24] INFO engine.py:156: server_args=ServerArgs(model_path='Qwen/Qwen2.5-1.5B-Instruct', tokenizer_path='Qwen/Qwen2.5-1.5B-Instruct', tokenizer_mode='auto', tokenizer_worker_num=1, skip_tokenizer_init=False, load_format='auto', model_loader_extra_config='{}', trust_remote_code=False, context_length=None, is_embedding=False, enable_multimodal=None, revision=None, model_impl='auto', host='127.0.0.1', port=30000, fastapi_root_path='', grpc_mode=False, skip_server_warmup=False, warmups=None, nccl_port=None, checkpoint_engine_wait_weights_before_ready=False, dtype='auto', quantization=None, quantization_param_path=None, kv_cache_dtype='auto', enable_fp32_lm_head=False, modelopt_quant=None, modelopt_checkpoint_restore_path=None, modelopt_checkpoint_save_path=None, modelopt_export_path=None, quantize_and_serve=False, rl_quant_profile=None, mem_fraction_static=0.7, max_running_requests=8, max_queued_requests=None, max_total_tokens=8192, chunked_prefill_size=8192, enable_dynami

...


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]



Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.43it/s]

Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.43it/s]




Capturing batches (bs=8 avail_mem=24.50 GB):   0%|          | 0/4 [00:00<?, ?it/s]


Capturing batches (bs=1 avail_mem=75.03 GB): 100%|██████████| 4/4 [00:19<00:00,  4.75s/it]


并发请求完成，总耗时: 1.90s

===== 样本 0 =====
 并发是指在同一时间点上，多个任务或进程同时执行。

===== 样本 1 =====
 异步是指在程序中，某些操作可以在另一个操作完成之前进行，而不需要等待其结果。异步编程允许程序在等待某些操作完成时继续执行其他任务。

===== 样本 2 =====
 当然，我可以帮你理解一下 async/await 的直觉比喻。

想象一下，你正在玩一个游戏，游戏中的任务是收集星星。但是，游戏的规则是，你不能在某个任务完成之前就开始下一个任务。这意味着，你不能在完成一个任务后立即开始下一个任务，因为你

===== 样本 3 =====
 大模型服务常见请求排队的原因主要有以下几点：

1. 大模型的计算资源有限：大模型需要大量的计算资源来训练和推理，而这些资源是有限的。因此，当多个请求同时到达时，服务需要优先处理那些具有更高优先级的请求，例如紧急或


In [3]:
# 释放资源：建议实验结束后执行

if "engine" in globals():
    shutdown_fn = getattr(engine, "shutdown", None)
    if callable(shutdown_fn):
        shutdown_fn()
        print("engine.shutdown() 已执行")
    else:
        print("当前版本未暴露 shutdown()，可直接重启 kernel 释放资源")

engine.shutdown() 已执行


## 常见报错与处理（1~2 条）

1. **`RuntimeError: ... event loop ...`（少见）**
   - 在 Notebook 中优先用 `await run_async_batch()`，不要再嵌套 `asyncio.run(...)`。
   - 若你改成 `.py` 脚本运行，再使用 `asyncio.run(...)`。

2. **OOM / 显存不足**
   - 降低 `max_running_requests`、`max_total_tokens`，并减少并发请求条数。
   - 或先换更小模型进行异步流程验证。

---